In [2]:
BASE_MODEL = "Qwen/Qwen3-4B"

# Use the parquet/native dataset (no scripts)
HF_ID = "gtfintechlab/financial_phrasebank_sentences_allagree"
HF_CONFIG = "5768"   # must specify one: ['5768','78516','944601']

LABEL_MAP = {0: "negative", 1: "neutral", 2: "positive"}
LABELS = ["negative", "neutral", "positive"]

N = 32
SEED = 0

MAX_LENGTH = 512
MAX_NEW_TOKENS = 2

# P-Tuning v2 settings
NUM_VIRTUAL_TOKENS = 20
ENCODER_HIDDEN_SIZE = 128
ENCODER_NUM_LAYERS = 2

# Training
MAX_STEPS = 200
TRAIN_BSZ = 2
GRAD_ACCUM = 8
LR = 2e-3

OUTPUT_DIR = f"ptuning_fpb_{BASE_MODEL.split('/')[-1]}_{HF_CONFIG}_N{N}_seed{SEED}"

In [3]:
!pip -q install -U transformers datasets accelerate peft evaluate scikit-learn gcsfs
import os, json, random, time
import numpy as np
import torch
from pathlib import Path

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 109.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 133.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.0/201.0 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 56.3 MB/s eta 0:00:00


In [4]:
!nvidia-smi
import torch, transformers
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

Mon Feb 23 19:12:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   35C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
BASE_MODEL = "Qwen/Qwen3-4B"

# Use the parquet/native dataset (no scripts)
HF_ID = "gtfintechlab/financial_phrasebank_sentences_allagree"
HF_CONFIG = "5768"   # must specify one: ['5768','78516','944601']

LABEL_MAP = {0: "negative", 1: "neutral", 2: "positive"}

N = 32
SEED = 0

MAX_LENGTH = 512
MAX_NEW_TOKENS = 2

# P-Tuning v2 settings
NUM_VIRTUAL_TOKENS = 20
ENCODER_HIDDEN_SIZE = 128
ENCODER_NUM_LAYERS = 2

# Training
MAX_STEPS = 200
TRAIN_BSZ = 2
GRAD_ACCUM = 8
LR = 2e-3

OUTPUT_DIR = f"ptuning_fpb_{BASE_MODEL.split('/')[-1]}_{HF_CONFIG}_N{N}_seed{SEED}"

In [6]:
from datasets import load_dataset, get_dataset_config_names
from collections import defaultdict
import random

ds = load_dataset(HF_ID, HF_CONFIG)

def to_examples(split):
    out = []
    for i, row in enumerate(ds[split]):
        y = LABEL_MAP[int(row["label"])]
        out.append({"id": f"{split}_{i}", "x": row["sentence"], "y": y, "label": y})
    return out

all_train = to_examples("train")
eval_set  = to_examples("test")  # fixed test split (recommended)

rng = random.Random(SEED)

# group by label
by_label = defaultdict(list)
for ex in all_train:
    by_label[ex["label"]].append(ex)

# shuffle within each label bucket
for lab in by_label:
    rng.shuffle(by_label[lab])

# round-robin interleave labels => stratified-ish global ordering
ordered = []
labels_sorted = sorted(by_label.keys())
i = 0
while True:
    progressed = False
    for lab in labels_sorted:
        if i < len(by_label[lab]):
            ordered.append(by_label[lab][i])
            progressed = True
    if not progressed:
        break
    i += 1

train_N = ordered[:N]

print("train_N:", len(train_N), "eval:", len(eval_set))
print("Label counts in train_N:", {lab: sum(ex["label"] == lab for ex in train_N) for lab in labels_sorted})
print("Example:", train_N[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

5768/train-00000-of-00001.parquet:   0%|          | 0.00/136k [00:00<?, ?B/s]

5768/test-00000-of-00001.parquet:   0%|          | 0.00/60.5k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1584 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/680 [00:00<?, ? examples/s]

train_N: 32 eval: 680
Label counts in train_N: {'negative': 11, 'neutral': 11, 'positive': 10}
Example: {'id': 'train_709', 'x': 'Net sales decreased to EUR 91.6 mn from EUR 109mn in the corresponding period in 2005 .', 'y': 'negative', 'label': 'negative'}


In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PromptEncoderConfig, get_peft_model, TaskType

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map="auto")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

peft_config = PromptEncoderConfig(
    peft_type="P_TUNING",
    task_type=TaskType.CAUSAL_LM,
    num_virtual_tokens=NUM_VIRTUAL_TOKENS,
    token_dim=model.config.hidden_size,
    num_layers=model.config.num_hidden_layers,
    num_attention_heads=model.config.num_attention_heads,
    encoder_hidden_size=ENCODER_HIDDEN_SIZE,
    encoder_num_layers=ENCODER_NUM_LAYERS,
)

pt_model = get_peft_model(model, peft_config)
pt_model.print_trainable_parameters()

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

trainable params: 725,760 || all params: 4,023,193,856 || trainable%: 0.0180


In [8]:
from datasets import Dataset
from transformers import DataCollatorForLanguageModeling, TrainingArguments, Trainer

def train_text(ex):
    return (
        "Task: Financial sentiment classification.\n"
        "Output EXACTLY one label: negative, neutral, or positive.\n\n"
        f"Input:\n{ex['x']}\n"
        f"Label: {ex['y']}"
    )

train_ds = Dataset.from_dict({"text": [train_text(ex) for ex in train_N]})

def tok(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH, padding="max_length")

train_tok = train_ds.map(tok, batched=True, remove_columns=["text"])
collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=TRAIN_BSZ,
    gradient_accumulation_steps=GRAD_ACCUM,
    max_steps=MAX_STEPS,
    learning_rate=LR,
    fp16=True,
    logging_steps=10,
    save_steps=MAX_STEPS,
    report_to=[],
)

trainer = Trainer(model=pt_model, args=args, train_dataset=train_tok, data_collator=collator)
trainer.train()
pt_model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved:", OUTPUT_DIR)

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Step,Training Loss
10,3.482237
20,2.230958
30,1.587081
40,1.491239
50,1.433710
60,1.379700
70,1.322286
80,1.247196
90,1.171070
100,1.082497


Saved: ptuning_fpb_Qwen3-4B_5768_N32_seed0


In [9]:
import time, numpy as np

LABELS = ["negative", "neutral", "positive"]

def parse_label(text):
    t = (text or "").strip().lower()
    for lab in LABELS:
        if lab in t.split(): return lab
    for lab in LABELS:
        if lab in t: return lab
    return "unknown"

def gpu_sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

@torch.no_grad()
def gen_one(x):
    prompt = (
        "Task: Financial sentiment classification.\n"
        "Output EXACTLY one label: negative, neutral, or positive.\n\n"
        f"Input:\n{x}\n"
        "Label:"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(pt_model.device)
    prompt_len = inputs["input_ids"].shape[-1]

    gpu_sync()
    t0 = time.perf_counter()
    out = pt_model.generate(**inputs, do_sample=False, max_new_tokens=MAX_NEW_TOKENS)
    gpu_sync()
    t1 = time.perf_counter()

    gen_ids = out[0][prompt_len:]
    text = tokenizer.decode(gen_ids, skip_special_tokens=True)
    return text, int(gen_ids.shape[-1]), (t1 - t0)

def evaluate(max_eval=300):
    pt_model.eval()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    lat = []
    total_new = 0
    total_t = 0.0
    correct = 0
    unk = 0

    for ex in eval_set[:max_eval]:
        txt, new_toks, dt = gen_one(ex["x"])
        pred = parse_label(txt)
        if pred == "unknown": unk += 1
        if pred == ex["y"]: correct += 1
        lat.append(dt * 1000)
        total_new += new_toks
        total_t += dt

    peak = (torch.cuda.max_memory_allocated()/(1024**2)) if torch.cuda.is_available() else None
    return {
        "accuracy": correct / min(max_eval, len(eval_set)),
        "unknown_frac": unk / min(max_eval, len(eval_set)),
        "lat_mean_ms": float(np.mean(lat)),
        "lat_p50_ms": float(np.percentile(lat, 50)),
        "lat_p95_ms": float(np.percentile(lat, 95)),
        "throughput_new_toks_per_s": (total_new / total_t) if total_t > 0 else None,
        "peak_vram_mb": peak,
    }

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:2141: UserWarning: Position ids are not supported for parameter efficient tuning. Ignoring position ids.
  warnings.warn("Position ids are not supported for parameter efficient tuning. Ignoring position ids.")


{'accuracy': 0.8066666666666666,
 'unknown_frac': 0.0033333333333333335,
 'lat_mean_ms': 118.86575406001612,
 'lat_p50_ms': 117.09605100031695,
 'lat_p95_ms': 128.32556784992448,
 'throughput_new_toks_per_s': 16.8257040542576,
 'peak_vram_mb': 7727.7880859375}

In [11]:
metrics = evaluate()

/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:2141: UserWarning: Position ids are not supported for parameter efficient tuning. Ignoring position ids.
  warnings.warn("Position ids are not supported for parameter efficient tuning. Ignoring position ids.")


In [12]:
import csv, os, platform, datetime
import torch

RESULTS_CSV = "results.csv"

row = {
    "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
    "base_model": BASE_MODEL,
    "hf_id": HF_ID,
    "hf_config": HF_CONFIG,
    "N": N,
    "seed": SEED,
    "max_length": MAX_LENGTH,
    "max_new_tokens": MAX_NEW_TOKENS,
    "max_steps": MAX_STEPS,
    "train_bsz": TRAIN_BSZ,
    "grad_accum": GRAD_ACCUM,
    "lr": LR,
    "num_virtual_tokens": NUM_VIRTUAL_TOKENS,
    "encoder_hidden_size": ENCODER_HIDDEN_SIZE,
    "encoder_num_layers": ENCODER_NUM_LAYERS,
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    **metrics,  # adds accuracy/latency/etc.
}

# Write header if file doesn't exist yet
write_header = not os.path.exists(RESULTS_CSV)

with open(RESULTS_CSV, "a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=row.keys())
    if write_header:
        writer.writeheader()
    writer.writerow(row)

print(f"Appended 1 row to {RESULTS_CSV}")

Appended 1 row to results.csv
